In [5]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

ko_text = "인생은 초콜릿이 든 박스와도 같다"
chinese_text = "生活就像一盒巧克力。"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 한국어 -> 영어
tokenizer.src_lang = "ko"
encoded_ko = tokenizer(ko_text, return_tensors="pt")
generated_ko = model.generate(**encoded_ko, forced_bos_token_id=tokenizer.get_lang_id("en"))
result_ko = tokenizer.batch_decode(generated_ko, skip_special_tokens=True)
print("Korean to English:", result_ko)

# 중국어 -> 영어
tokenizer.src_lang = "zh"
encoded_zh = tokenizer(chinese_text, return_tensors="pt")
generated_zh = model.generate(**encoded_zh, forced_bos_token_id=tokenizer.get_lang_id("en"))
result_zh = tokenizer.batch_decode(generated_zh, skip_special_tokens=True)
print("Chinese to English:", result_zh)

Korean to English: ['Life is like a chocolate box.']
Chinese to English: ['Life is like a box of chocolate.']


In [6]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# 모델 및 토크나이저 로드
model_name = "facebook/m2m100_418M"
model = M2M100ForConditionalGeneration.from_pretrained(model_name)
tokenizer = M2M100Tokenizer.from_pretrained(model_name)

# 지원 언어 목록 (M2M100 지원 언어 일부)
language_dict = {
    "Korean": "ko",
    "English": "en",
    "Chinese": "zh",
    "French": "fr",
    "German": "de",
    "Japanese": "ja",
    "Spanish": "es"
}

def translate_text(text, source_lang, target_lang):
    if not text.strip():
        return ""

    src_code = language_dict[source_lang]
    tgt_code = language_dict[target_lang]

    tokenizer.src_lang = src_code
    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(**encoded, forced_bos_token_id=tokenizer.get_lang_id(tgt_code))
    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return result[0]

# Gradio UI 구성
with gr.Blocks() as demo:
    gr.Markdown("## 🌍 M2M100 다국어 번역기")

    with gr.Row():
        source_lang = gr.Dropdown(label="Source Language", choices=list(language_dict.keys()), value="Korean")
        target_lang = gr.Dropdown(label="Target Language", choices=list(language_dict.keys()), value="English")

    with gr.Row():
        input_text = gr.Textbox(label="Input Text", lines=5, placeholder="번역할 문장을 입력하세요.")
        output_text = gr.Textbox(label="Translated Text", lines=5, interactive=False)

    translate_btn = gr.Button("Translate")

    translate_btn.click(
        fn=translate_text,
        inputs=[input_text, source_lang, target_lang],
        outputs=output_text
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
demo.close()

Closing server running on port: 7860


In [10]:
# !pip install datasets soundfile

In [11]:
import gradio as gr
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
)
from datasets import load_dataset
import torch
import soundfile as sf
import os
import uuid

# ----------------- Load translation model -----------------
translation_model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
translation_tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# ----------------- Load TTS model -----------------
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
tts_vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
tts_embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(tts_embeddings_dataset[7306]["xvector"]).unsqueeze(0)

# ----------------- Supported Languages -----------------
language_dict = {
    "Korean": "ko",
    "English": "en",
    "Chinese": "zh",
    "French": "fr",
    "German": "de",
    "Japanese": "ja",
    "Spanish": "es"
}

# ----------------- Translation + Optional TTS -----------------
def translate_and_tts(text, source_lang, target_lang):
    if not text.strip():
        return "", None

    # 1. 번역
    src_code = language_dict[source_lang]
    tgt_code = language_dict[target_lang]
    translation_tokenizer.src_lang = src_code
    encoded = translation_tokenizer(text, return_tensors="pt")
    generated_tokens = translation_model.generate(
        **encoded,
        forced_bos_token_id=translation_tokenizer.get_lang_id(tgt_code)
    )
    translated_text = translation_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    # 2. 영어일 경우 음성 생성
    audio_path = None
    if tgt_code == "en":
        inputs = tts_processor(text=translated_text, return_tensors="pt")
        speech = tts_model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=tts_vocoder)

        # 고유 파일명 생성
        audio_path = f"speech_{uuid.uuid4().hex[:8]}.wav"
        sf.write(audio_path, speech.numpy(), samplerate=16000)

    return translated_text, audio_path

# ----------------- Gradio UI -----------------
with gr.Blocks() as demo:
    gr.Markdown("## 🌍 다국어 번역기 + 영어 TTS")

    with gr.Row():
        source_lang = gr.Dropdown(label="Source Language", choices=list(language_dict.keys()), value="Korean")
        target_lang = gr.Dropdown(label="Target Language", choices=list(language_dict.keys()), value="English")

    with gr.Row():
        input_text = gr.Textbox(label="Input Text", lines=5, placeholder="번역할 문장을 입력하세요.")
        output_text = gr.Textbox(label="Translated Text", lines=5, interactive=False)
    
    audio_output = gr.Audio(label="English Speech Output", interactive=False)
    translate_btn = gr.Button("Translate")

    translate_btn.click(
        fn=translate_and_tts,
        inputs=[input_text, source_lang, target_lang],
        outputs=[output_text, audio_output]
    )

demo.launch()

C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--microsoft--speecht5_tts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWa

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
